# Esperienza 6

In [1]:
import os
import ROOT
import numpy as np
import pandas as pd

ROOT.gStyle.SetOptFit(1111)    # label con chi2/ndf, probabilita e parametri
ROOT.gStyle.SetFitFormat("4.2e")
ROOT.gStyle.SetStatX(0.67)  # Coordinata X dell'angolo in alto a destra
ROOT.gStyle.SetStatY(0.88)  # Coordinata Y dell'angolo in alto a destra
ROOT.gStyle.SetStatW(0.15)  # Larghezza
ROOT.gStyle.SetStatH(0.10)  # Altezza
os.makedirs("Immagini", exist_ok=True)

## Dati 

In [2]:
S = 4040.0          # distanza fotodiodo-fenditura [mm]
err_S = 10.0        # errore su S [mm]
lam_v = 532.0 * 1e-6   # laser verde [mm]
lam_r = 650.0 * 1e-6   #  laser rosso [mm]

# PRIMA PARTE

## Lettura dati

In [3]:
#x, y, sigma_x, sigma_y
raw = pd.read_excel("Esperienza6c.xlsx", sheet_name="Dati codice", header=None)

verde = raw.iloc[6:66, 4:8].astype("float64").to_numpy()    #  E7:H87
rosso = raw.iloc[6:66, 15:19].astype("float64").to_numpy()   #  J7:M67

# ordino per x crescente
verde = np.ascontiguousarray(verde[np.argsort(verde[:, 0])].T)
rosso = np.ascontiguousarray(rosso[np.argsort(rosso[:, 0])].T)
 

x_v, y_v, err_x_v, err_y_v = verde
x_r, y_r, err_x_r, err_y_r = rosso


cut = 5
idx = np.r_[0:cut, -cut:0]

x_r, y_r, err_x_r, err_y_r = [np.delete(np.asarray(a), idx) for a in (x_r, y_r, err_x_r, err_y_r)]
rosso = (x_r, y_r, err_x_r, err_y_r)

x_v, y_v, err_x_v, err_y_v = [np.delete(np.asarray(a), idx) for a in (x_v, y_v, err_x_v, err_y_v)]
verde = (x_v, y_v, err_x_v, err_y_v)


'''
x_r = np.array(x_r, dtype = 'float')
x_r[34:] -= 0.3

rosso = (x_r, y_r, err_x_r, err_y_r)
'''


"\nx_r = np.array(x_r, dtype = 'float')\nx_r[34:] -= 0.3\n\nrosso = (x_r, y_r, err_x_r, err_y_r)\n"

## Funzioni

Modello: $y = A\left[\dfrac{\sin(B(x-C))}{B(x-C)}\right]^2 + D$

In [4]:
def sinc2(x, par):
    t = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    D = par[3]
    return A * np.sinc(B * (t - C) / np.pi)**2 + D      # np.sinc(z) = sin(pi z)/(pi z)

def modellopy(t, A, B, C, D):
    return A * np.sinc(B * (t - C) / np.pi)**2 + D

def derivata(x, par):
    t = x
    dt = 1e-6
    return (modellopy(t + dt/2, par[0], par[1], par[2], par[3]) - modellopy(t - dt/2, par[0], par[1], par[2], par[3])) / dt

def retta(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

## Fit sinc² - laser verde

In [5]:
# fit  con solo err_y 
pre_v = ROOT.TF1("pre_v", sinc2, min(x_v), max(x_v), 4)
pre_v.SetParameters(y_v.max(), 0.650, 0.3, 0)
gr_pre_v = ROOT.TGraphErrors(len(x_v), x_v, y_v, np.zeros_like(x_v), err_y_v)
gr_pre_v.Fit(pre_v, "SQ0")


# fit completo con ROOT
SincFit_v = ROOT.TF1("model_v", sinc2, min(x_v), max(x_v), 4)
SincFit_v.SetParameters(pre_v.GetParameter(0), pre_v.GetParameter(1), pre_v.GetParameter(2), pre_v.GetParameter(3))
SincFit_v.SetParNames("A", "B", "C", "D")
SincFit_v.SetParLimits(0, 0.0, 2.8)
#SincFit_v.SetParLimits(3, 0.0, 30000000000)
SincFit_v.SetNpx(5000)   

gr1_v = ROOT.TGraphErrors(len(x_v), x_v, y_v, err_x_v, err_y_v)
gr1_v.SetTitle("Dati e fit (verde); x [mm]; V [V]")
gr1_v.SetMarkerStyle(20)
gr1_v.SetMarkerSize(0.7)
gr1_v.SetMarkerColor(ROOT.kAzure + 2)
gr1_v.SetLineColor(ROOT.kAzure + 3)

fit_res_v = gr1_v.Fit(SincFit_v, "S")
print(f"Fit status: {fit_res_v.Status()}")

par_v = [SincFit_v.GetParameter(i) for i in range(4)]
errors_v = [SincFit_v.GetParError(i) for i in range(4)]
chi2_v = fit_res_v.Chi2()
ndf_v = fit_res_v.Ndf()

err_yeff_v = np.sqrt(err_y_v**2 + (derivata(x_v, par_v) * err_x_v)**2)   # errore effettivo

print("Parametri stimati:")
print(f"A = {par_v[0]} ± {errors_v[0]}")
print(f"B = {par_v[1]} ± {errors_v[1]}")
print(f"C = {par_v[2]} ± {errors_v[2]}")
print(f"D = {par_v[3]} ± {errors_v[3]}")
print(f"Chi2 = {chi2_v}, ndf = {ndf_v}")

c1_v = ROOT.TCanvas("c1_v", "Fit (verde)", 800, 600)
c1_v.SetGrid()
 
gr1_v.Draw("AP")


c1_v.Draw()
c1_v.SaveAs("Immagini/verde_fit.png")

Fit status: 0
Parametri stimati:
A = 2.799999999951636 ± 0.006136934891514345
B = 0.6432537347894511 ± 0.0003019999218258401
C = 0.36794865126419074 ± 0.007674856786680988
D = 1.5385507214230766e-05 ± 5.051500330597984e-05
Chi2 = 696.8073994751453, ndf = 46
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      696.807
NDf                       =           46
Edm                       =   1.1055e-07
NCalls                    =          224
A                         =          2.8   +/-   0.00613693   	 (limited)
B                         =     0.643254   +/-   0.000302    
C                         =     0.367949   +/-   0.00767486  
D                         =  1.53855e-05   +/-   5.0515e-05  


Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 
Info in <TCanvas::Print>: png file Immagini/verde_fit.png has been created


In [6]:
if np.abs(par_v[2]) < 3*errors_v[2]:
    print("C è compatibile con 0")
else:
    print("C NON è compatibile con 0")

print("---")

if np.abs(par_v[3]) < 3*errors_v[3]:
    print("D è compatibile con 0")
else:
    print("D NON è compatibile con 0")

C NON è compatibile con 0
---
D è compatibile con 0


## Fit sinc² - laser rosso

In [7]:
# fit preliminare con solo err_y (serve per partire da buoni parametri iniziali)
pre_r = ROOT.TF1("pre_r", sinc2, min(x_r), max(x_r), 4)
pre_r.SetParameters(y_r.max(), 0.52, -1.4, 0)
gr_pre_r = ROOT.TGraphErrors(len(x_r), x_r, y_r, np.zeros_like(x_r), err_y_r)
gr_pre_r.Fit(pre_r, "SQ0")

# fit completo con ROOT
SincFit_r = ROOT.TF1("model_r", sinc2, min(x_r), max(x_r), 4)
SincFit_r.SetParameters(pre_r.GetParameter(0), pre_r.GetParameter(1), pre_r.GetParameter(2), pre_r.GetParameter(3))
SincFit_r.SetParNames("A", "B", "C", "D")
SincFit_r.SetParLimits(0, 0.90, 1.0)
SincFit_r.SetParLimits(2, -1.3, -1.1)
#SincFit_r.SetParLimits(3, 0.0, 30000000000)
SincFit_r.SetNpx(5000)   

gr1_r = ROOT.TGraphErrors(len(x_r), x_r, y_r, err_x_r, err_y_r)
gr1_r.SetTitle("Dati e fit (rosso); x [mm]; V [V]")
gr1_r.SetMarkerStyle(20)
gr1_r.SetMarkerSize(0.7)
gr1_r.SetMarkerColor(ROOT.kAzure + 2)
gr1_r.SetLineColor(ROOT.kAzure + 3)

fit_res_r = gr1_r.Fit(SincFit_r, "SR")
print(f"Fit status: {fit_res_r.Status()}")

par_r = [SincFit_r.GetParameter(i) for i in range(4)]
errors_r = [SincFit_r.GetParError(i) for i in range(4)]
chi2_r = fit_res_r.Chi2()
ndf_r = fit_res_r.Ndf()

err_yeff_r = np.sqrt(err_y_r**2 + (derivata(x_r, par_r) * err_x_r)**2)   # errore effettivo

print("Parametri stimati:")
print(f"A = {par_r[0]} ± {errors_r[0]}")
print(f"B = {par_r[1]} ± {errors_r[1]}")
print(f"C = {par_r[2]} ± {errors_r[2]}")
print(f"D = {par_r[3]} ± {errors_r[3]}")
print(f"Chi2 = {chi2_r}, ndf = {ndf_r}")

c1_r = ROOT.TCanvas("c1_r", "Fit (rosso)", 800, 600)
c1_r.SetGrid()

gr1_r.Draw("AP")




c1_r.Draw()
c1_r.SaveAs("Immagini/rosso_fit.png")

Fit status: 0
Parametri stimati:
A = 0.9058472241815112 ± 0.0011504505412546218
B = 0.5059317312251478 ± 0.00010979273670786506
C = -1.100000000134229 ± 8.949722149342865e-05
D = -0.0019051621735382649 ± 3.665909155357243e-05
Chi2 = 7442.646169161529, ndf = 46
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      7442.65
NDf                       =           46
Edm                       =  8.07217e-06
NCalls                    =          169
A                         =     0.905847   +/-   0.00115045   	 (limited)
B                         =     0.505932   +/-   0.000109793 
C                         =         -1.1   +/-   8.94972e-05  	 (limited)
D                         =  -0.00190516   +/-   3.66591e-05 


Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 
Info in <TCanvas::Print>: png file Immagini/rosso_fit.png has been created


In [8]:
i = 0                                    # dataset rosso da scansionare
x_r, y_r, err_x_r, err_y_r = rosso
best = SincFit_r

p_opt = [best.GetParameter(j) for j in range(4)]
C_opt = -1.44
sC    = best.GetParError(0)
C_vals = np.linspace(C_opt - 0.4, C_opt + 0.5, 5000)

# TF1 di lavoro per valutare il modello (stessa sinc2 del fit)
f_eval = ROOT.TF1(f"eval_{i}", sinc2, min(x_r), max(x_r), 4)

chi2_slice = np.zeros_like(C_vals)
for k, C in enumerate(C_vals):
    p = [p_opt[0], p_opt[1], C, p_opt[3]]          # solo C varia
    f_eval.SetParameters(*p)
    y_mod = np.array([f_eval.Eval(xx) for xx in x_r])
    # errore efficace ricalcolato con il C di prova
    w = 1.0 / (err_y_r**2 + (derivata(x_r, p) * err_x_r)**2)
    chi2_slice[k] = np.sum(w * (y_r - y_mod)**2)

g_slice = ROOT.TGraph(len(C_vals), C_vals, chi2_slice)
g_slice.SetTitle(f"Slice di #chi^{{2}} in A; A[V]; #chi^{{2}}")
g_slice.SetLineColor(ROOT.kRed)
g_slice.SetLineWidth(2)

c_slice = ROOT.TCanvas("c_slice", "chi2 slice vs C", 800, 600)
c_slice.SetGrid()
# c_slice.SetLogy()                      # decommenta se il minimo si perde
g_slice.Draw("AL")

chi2_min = chi2_slice.min()
line = ROOT.TLine(C_vals[0], chi2_min + 1, C_vals[-1], chi2_min + 1)
line.SetLineStyle(2)
line.Draw()                              # soglia Δχ² = 1

c_slice.Draw()
c_slice.SaveAs(f"Immagini/chi2_slice_C_{i}.png")

# intervallo 1σ dalla slice (punti con Δχ² < 1)
inside = C_vals[chi2_slice < chi2_min + 1]
print(f"C_min = {C_vals[np.argmin(chi2_slice)]:.5f}, "
      f"intervallo 1σ: [{inside.min():.5f}, {inside.max():.5f}]")

C_min = -1.08061, intervallo 1σ: [-1.08241, -1.07881]


Info in <TCanvas::Print>: png file Immagini/chi2_slice_C_0.png has been created


In [9]:
'''
n0 = 32                                   # primo indice traslato (dopo la misura 35)
offsets = np.linspace(-2, 2, 81)         # passo 0.05 mm

x0 = np.array(x_r, dtype=float)           # copia dei dati originali (non modificarli)
y0 = np.asarray(y_r, dtype=float)
ex0 = np.asarray(err_x_r, dtype=float)
ey0 = np.asarray(err_y_r, dtype=float)

def fit_con_offset(off, nome, quiet=True):
    """Fit completo (pre-fit + fit con err_x) con x[n0:] += off. Ritorna (chi2, ndf, status, f, gr)."""
    x = x0.copy()
    x[n0:] += off

    # pre-fit con solo err_y
    pre = ROOT.TF1(f"pre_{nome}", sinc2, x.min(), x.max(), 4)
    pre.SetParameters(y0.max(), 0.52, -1.4, 0)
    g_pre = ROOT.TGraphErrors(len(x), x, y0, np.zeros_like(x), ey0)
    g_pre.Fit(pre, "SQ0")

    # fit completo
    f = ROOT.TF1(f"model_{nome}", sinc2, x.min(), x.max(), 4)
    f.SetParameters(*[pre.GetParameter(k) for k in range(4)])
    f.SetParNames("A", "B", "C", "D")
    f.SetParLimits(0, 0.0, 1.0)
    #f.SetParLimits(2, -2, -1.3)
    #f.SetParLimits(3, 0.0, 30000000000)
    f.SetNpx(5000)

    gr = ROOT.TGraphErrors(len(x), x, y0, ex0 if False else err_x_scan(x, ex0), ey0)
    res = gr.Fit(f, "SQN" if quiet else "S")
    return res.Chi2(), res.Ndf(), res.Status(), f, gr, pre, g_pre

def err_x_scan(x, ex):
    return ex                              # err_x invariato dalla traslazione

# ---- scansione ----
chi2_scan = np.full(len(offsets), np.nan)
for k, off in enumerate(offsets):
    chi2, ndf, status, *_ = fit_con_offset(off, "scan")
    chi2_scan[k] = chi2 if status == 0 else np.nan     # scarta fit non convergiti

k_best = np.nanargmin(chi2_scan)
off_best = offsets[k_best]
print(f"offset migliore = {off_best:.3f} mm, chi2 = {chi2_scan[k_best]:.2f}, ndf = {ndf}")

# ---- rifit al best offset e plot solo di quello ----
chi2_r, ndf_r, st, SincFit_r, gr1_r, pre_r, gr_pre_r = fit_con_offset(off_best, "best", quiet=False)
gr1_r.SetTitle(f"Dati e fit (rosso), offset = {off_best:.2f} mm; x [mm]; V [V]")
gr1_r.SetMarkerStyle(20)
gr1_r.SetMarkerSize(0.7)
gr1_r.SetMarkerColor(ROOT.kAzure + 2)
gr1_r.SetLineColor(ROOT.kAzure + 3)

par_r = [SincFit_r.GetParameter(i) for i in range(4)]
errors_r = [SincFit_r.GetParError(i) for i in range(4)]
for n, p, e in zip("ABCD", par_r, errors_r):
    print(f"{n} = {p} ± {e}")
print(f"Chi2 = {chi2_r}, ndf = {ndf_r}")

c1_r = ROOT.TCanvas("c1_r_best", "Fit (rosso) best offset", 800, 600)
c1_r.SetGrid()
gr1_r.Draw("AP")
SincFit_r.Draw("same")
c1_r.Draw()
c1_r.SaveAs("Immagini/rosso_fit_best_offset.png")

# aggiorna i dati con l'offset ottimale
x_r = x0.copy(); x_r[n0:] += off_best
rosso = (x_r, y0, ex0, ey0)
'''

'\nn0 = 32                                   # primo indice traslato (dopo la misura 35)\noffsets = np.linspace(-2, 2, 81)         # passo 0.05 mm\n\nx0 = np.array(x_r, dtype=float)           # copia dei dati originali (non modificarli)\ny0 = np.asarray(y_r, dtype=float)\nex0 = np.asarray(err_x_r, dtype=float)\ney0 = np.asarray(err_y_r, dtype=float)\n\ndef fit_con_offset(off, nome, quiet=True):\n    """Fit completo (pre-fit + fit con err_x) con x[n0:] += off. Ritorna (chi2, ndf, status, f, gr)."""\n    x = x0.copy()\n    x[n0:] += off\n\n    # pre-fit con solo err_y\n    pre = ROOT.TF1(f"pre_{nome}", sinc2, x.min(), x.max(), 4)\n    pre.SetParameters(y0.max(), 0.52, -1.4, 0)\n    g_pre = ROOT.TGraphErrors(len(x), x, y0, np.zeros_like(x), ey0)\n    g_pre.Fit(pre, "SQ0")\n\n    # fit completo\n    f = ROOT.TF1(f"model_{nome}", sinc2, x.min(), x.max(), 4)\n    f.SetParameters(*[pre.GetParameter(k) for k in range(4)])\n    f.SetParNames("A", "B", "C", "D")\n    f.SetParLimits(0, 0.0, 1.0)\

In [10]:
if np.abs(par_r[2]) < 3*errors_r[2]:
    print("C è compatibile con 0")
else:
    print("C NON è compatibile con 0")

print("---")

if np.abs(par_r[3]) < 3*errors_r[3]:
    print("D è compatibile con 0")
else:
    print("D NON è compatibile con 0")


C NON è compatibile con 0
---
D NON è compatibile con 0


## Larghezza della fenditura d

$d = \dfrac{B\lambda S}{2\pi}$, $\quad\sigma_d = \dfrac{\lambda}{2\pi}\sqrt{(S\,\sigma_B)^2 + (B\,\sigma_S)^2}$

In [11]:
d_v = abs(par_v[1]) * lam_v * S / (2*np.pi)
err_d_v = lam_v / (2 * np.pi) * np.sqrt((S * errors_v[1])**2 + (par_v[1] * err_S)**2)

d_r = abs(par_r[1]) * lam_r * S / (2*np.pi)
err_d_r = lam_r / (2 * np.pi) * np.sqrt((S * errors_r[1])**2 + (par_r[1] * err_S)**2)



print(f"d verde = {d_v} ± {err_d_v} mm")
print(f"d rosso = {d_r} ± {err_d_r} mm")

print(f"la larghezza della fenditura verde è 2d = {2*d_v} ± {2*err_d_v} mm")
print(f"la larghezza della fenditura rossa è 2d = {2*d_r} ± {2*err_d_v} mm")

d verde = 0.22003686339291917 ± 0.0005543561988560995 mm
d rosso = 0.211449553251138 ± 0.00052539763963358 mm
la larghezza della fenditura verde è 2d = 0.44007372678583834 ± 0.001108712397712199 mm
la larghezza della fenditura rossa è 2d = 0.422899106502276 ± 0.001108712397712199 mm


## Compatibilità tra le due d

$|d_{verde} - d_{rosso}| < 3\sqrt{\sigma_{d,verde}^2 + \sigma_{d,rosso}^2}$

In [12]:
diff_d = np.abs(d_v - d_r)
err_diff_d = np.sqrt(err_d_v**2 + err_d_r**2)
print(err_diff_d)

print(f"d verde = {d_v} ± {err_d_v} mm")
print(f"d rosso = {d_r} ± {err_d_r} mm")
print(f"|d verde - d rosso| = {diff_d} mm  ({diff_d / err_diff_d:.2f} sigma)")

if diff_d < 3*err_diff_d:
    print("d verde è compatibile con d rosso")
else:
    print("d verde NON è compatibile con d rosso")

0.0007637758014906734
d verde = 0.22003686339291917 ± 0.0005543561988560995 mm
d rosso = 0.211449553251138 ± 0.00052539763963358 mm
|d verde - d rosso| = 0.008587310141781157 mm  (11.24 sigma)
d verde NON è compatibile con d rosso


## Residui normalizzati - laser verde

In [13]:
y_attesi_v = np.array([SincFit_v.Eval(x_val) for x_val in x_v])
residuals_v = y_v - y_attesi_v
residuals_norm_v = residuals_v / err_yeff_v

gaussian_v = ROOT.TF1("gaussian_v", "gaus", min(residuals_norm_v), max(residuals_norm_v), 3)
gaussian_v.SetParNames("const", "mu", "sigma")
gaussian_v.SetLineColor(ROOT.kRed + 1)
gaussian_v.SetLineWidth(3)

hist_v = ROOT.TH1F("hist_v", "Istogramma residui normalizzati (verde)", 10, -12, 12)
for val in residuals_norm_v:
    hist_v.Fill(val)

gaussian_v.SetParameters(hist_v.GetMaximum(), hist_v.GetMean(), hist_v.GetRMS())

hist_v.SetLineColor(ROOT.kAzure + 3)
hist_v.SetLineWidth(2)
hist_v.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)
hist_v.GetXaxis().SetRangeUser(-12, 12)
hist_v.GetXaxis().SetTitle("residuo normalizzato")
hist_v.GetYaxis().SetTitle("conteggi")

r_v = hist_v.Fit(gaussian_v, "S")
print(f"Fit status: {r_v.Status()}")

par_g_v = [gaussian_v.GetParameter(i) for i in range(3)]
err_g_v = [gaussian_v.GetParError(i) for i in range(3)]

print("Parametri stimati:")
print(f"const = {par_g_v[0]} ± {err_g_v[0]}")
print(f"mu = {par_g_v[1]} ± {err_g_v[1]}")
print(f"sigma = {par_g_v[2]} ± {err_g_v[2]}")

c2_v = ROOT.TCanvas("c2_v", "Istogramma (verde)", 800, 600)
c2_v.SetGrid()

hist_v.Draw()


c2_v.Draw()
c2_v.SaveAs("Immagini/verde_hist.png")

Fit status: 0
Parametri stimati:
const = 13.917531056748883 ± 2.8925506505260477
mu = 1.721865892061702 ± 0.5539319970413463
sigma = 2.712571573167584 ± 0.4131602330569357
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      10.2946
NDf                       =            4
Edm                       =  7.21579e-08
NCalls                    =           86
const                     =      13.9175   +/-   2.89255     
mu                        =      1.72187   +/-   0.553932    
sigma                     =      2.71257   +/-   0.41316      	 (limited)


Info in <TCanvas::Print>: png file Immagini/verde_hist.png has been created


In [14]:
if np.abs(par_g_v[1]) < 3*err_g_v[1]:
    print("mu è compatibile con 0")
else:
    print("mu NON è compatibile con 0")

print("---")

if np.abs(par_g_v[2] - 1) < 3*err_g_v[2]:
    print("sigma è compatibile con 1")
else:
    print("sigma NON è compatibile con 1")

mu NON è compatibile con 0
---
sigma NON è compatibile con 1


## Residui normalizzati - laser rosso

In [15]:
y_attesi_r = np.array([SincFit_r.Eval(x_val) for x_val in x_r])
residuals_r = y_r - y_attesi_r
residuals_norm_r = residuals_r / err_yeff_r

dev_std = np.std(residuals_norm_r)
delta_bin = 3.5 * dev_std / (len(residuals_norm_r))**(1/3)
n_bin= (np.max(residuals_norm_r) - np.min(residuals_norm_r))/delta_bin
n_bin = int(np.round(n_bin))
print("numero di bin senza code", n_bin)

gaussian_r = ROOT.TF1("gaussian_r", "gaus", min(residuals_norm_r), max(residuals_norm_r), 3)
gaussian_r.SetParNames("const", "mu", "sigma")
gaussian_r.SetLineColor(ROOT.kRed + 1)
gaussian_r.SetLineWidth(3)

hist_r = ROOT.TH1F("hist_r", "Istogramma residui normalizzati (rosso)", 6, -20, 20)
for val in residuals_norm_r:
    hist_r.Fill(val)

gaussian_r.SetParameters(hist_r.GetMaximum(), hist_r.GetMean(), hist_r.GetRMS())

hist_r.SetLineColor(ROOT.kAzure + 3)
hist_r.SetLineWidth(2)
hist_r.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)
hist_r.GetXaxis().SetTitle("residuo normalizzato")
hist_r.GetXaxis().SetRangeUser(-16, 16)
hist_r.GetYaxis().SetTitle("conteggi")

r_r = hist_r.Fit(gaussian_r, "S")
print(f"Fit status: {r_r.Status()}")

par_g_r = [gaussian_r.GetParameter(i) for i in range(3)]
err_g_r = [gaussian_r.GetParError(i) for i in range(3)]

print("Parametri stimati:")
print(f"const = {par_g_r[0]} ± {err_g_r[0]}")
print(f"mu = {par_g_r[1]} ± {err_g_r[1]}")
print(f"sigma = {par_g_r[2]} ± {err_g_r[2]}")

c2_r = ROOT.TCanvas("c2_r", "Istogramma (rosso)", 800, 600)
c2_r.SetGrid()

hist_r.Draw()


c2_r.Draw()
c2_r.SaveAs("Immagini/rosso_hist.png")

numero di bin senza code 6
Fit status: 0
Parametri stimati:
const = 15.3070865593768 ± 3.557886851369285
mu = 0.8707585375261009 ± 1.3297697577952756
sigma = 7.76376317653883 ± 1.651967242151993
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1.86741
NDf                       =            2
Edm                       =   9.3306e-06
NCalls                    =           53
const                     =      15.3071   +/-   3.55789     
mu                        =     0.870759   +/-   1.32977     
sigma                     =      7.76376   +/-   1.65197      	 (limited)


Info in <TCanvas::Print>: png file Immagini/rosso_hist.png has been created


In [16]:
if np.abs(par_g_r[1]) < 3*err_g_r[1]:
    print("mu è compatibile con 0")
else:
    print("mu NON è compatibile con 0")

print("---")

if np.abs(par_g_r[2] - 1) < 3*err_g_r[2]:
    print("sigma è compatibile con 1")
else:
    print("sigma NON è compatibile con 1")

mu è compatibile con 0
---
sigma NON è compatibile con 1


## Residui in funzione di x - laser verde

In [17]:
line_v = ROOT.TF1("line_v", retta, min(x_v), max(x_v), 2)
line_v.SetParameters(0, 0)
line_v.SetParNames("m", "q")
line_v.SetLineColor(ROOT.kRed + 1)
line_v.SetLineWidth(2)

gr2_v = ROOT.TGraphErrors(len(x_v), x_v, residuals_v, np.zeros_like(x_v), err_yeff_v)
gr2_v.SetTitle("Residui in funzione di x (verde); x [mm]; V_{misurato} - V_{fit} [V]")
gr2_v.SetMarkerStyle(20)
gr2_v.SetMarkerSize(0.7)
gr2_v.SetMarkerColor(ROOT.kAzure + 2)
gr2_v.SetLineColor(ROOT.kAzure + 3)

fit_res2_v = gr2_v.Fit(line_v, "S")
print(f"Fit status: {fit_res2_v.Status()}")

par_l_v = [line_v.GetParameter(i) for i in range(2)]
err_l_v = [line_v.GetParError(i) for i in range(2)]

print("Parametri stimati:")
print(f"m = {par_l_v[0]} ± {err_l_v[0]}")
print(f"q = {par_l_v[1]} ± {err_l_v[1]}")
print(f"Chi2 = {fit_res2_v.Chi2()}")

if np.abs(par_l_v[0]) < 3*err_l_v[0]:
    print("m è compatibile con 0")
else:
    print("m NON è compatibile con 0")

if np.abs(par_l_v[1]) < 3*err_l_v[1]:
    print("q è compatibile con 0")
else:
    print("q NON è compatibile con 0")

c3_v = ROOT.TCanvas("c3_v", "Residui (verde)", 800, 600)
c3_v.SetGrid()

gr2_v.Draw("AP")



c3_v.Draw()
c3_v.SaveAs("Immagini/verde_residui.png")

Fit status: 0
Parametri stimati:
m = -8.073729549578447e-06 ± 2.516564796836716e-06
q = -3.442474417498735e-07 ± 5.03069139504019e-05
Chi2 = 686.5146341327519
m NON è compatibile con 0
q è compatibile con 0
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      686.515
NDf                       =           48
Edm                       =  4.49761e-15
NCalls                    =           33
m                         = -8.07373e-06   +/-   2.51656e-06 
q                         = -3.44247e-07   +/-   5.03069e-05 


Info in <TCanvas::Print>: png file Immagini/verde_residui.png has been created


## Residui in funzione di x - laser rosso

In [18]:
line_r = ROOT.TF1("line_r", retta, min(x_r), max(x_r), 2)
line_r.SetParameters(0, 0)
line_r.SetParNames("m", "q")
line_r.SetLineColor(ROOT.kRed + 1)
line_r.SetLineWidth(2)

gr2_r = ROOT.TGraphErrors(len(x_r), x_r, residuals_r, np.zeros_like(x_r), err_yeff_r)
gr2_r.SetTitle("Residui in funzione di x (rosso); x [mm]; V_{misurato} - V_{fit} [V]")
gr2_r.SetMarkerStyle(20)
gr2_r.SetMarkerSize(0.7)
gr2_r.SetMarkerColor(ROOT.kAzure + 2)
gr2_r.SetLineColor(ROOT.kAzure + 3)

fit_res2_r = gr2_r.Fit(line_r, "S")
print(f"Fit status: {fit_res2_r.Status()}")

par_l_r = [line_r.GetParameter(i) for i in range(2)]
err_l_r = [line_r.GetParError(i) for i in range(2)]

print("Parametri stimati:")
print(f"m = {par_l_r[0]} ± {err_l_r[0]}")
print(f"q = {par_l_r[1]} ± {err_l_r[1]}")
print(f"Chi2 = {fit_res2_r.Chi2()}")

if np.abs(par_l_r[0]) < 3*err_l_r[0]:
    print("m è compatibile con 0")
else:
    print("m NON è compatibile con 0")

if np.abs(par_l_r[1]) < 3*err_l_r[1]:
    print("q è compatibile con 0")
else:
    print("q NON è compatibile con 0")

c3_r = ROOT.TCanvas("c3_r", "Residui (rosso)", 800, 600)
c3_r.SetGrid()

gr2_r.Draw("AP")



c3_r.Draw()
c3_r.SaveAs("Immagini/rosso_residui.png")

Fit status: 0
Parametri stimati:
m = -1.5698810635059205e-06 ± 1.79977691637908e-06
q = 1.7757514948790595e-06 ± 3.4153988812590805e-05
Chi2 = 7441.885297207839
m è compatibile con 0
q è compatibile con 0
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      7441.89
NDf                       =           48
Edm                       =  1.06349e-07
NCalls                    =           33
m                         = -1.56988e-06   +/-   1.79978e-06 
q                         =  1.77575e-06   +/-   3.4154e-05  


Info in <TCanvas::Print>: png file Immagini/rosso_residui.png has been created


## Shift tra i due laser

Primo minimo è $x_1 = \pi/B = \dfrac{\lambda S}{2d}$, quindi $\dfrac{x_{1,rosso}}{x_{1,verde}} = \dfrac{\lambda_{rosso}}{\lambda_{verde}}$. Lo shift dei minimi deve corrispondere a $\Delta\lambda = \lambda_{rosso} - \lambda_{verde}$.

In [19]:
# posizione del primo minimo rispetto al massimo centrale
x1_v = np.pi / abs(par_v[1])
err_x1_v = np.pi * errors_v[1] / par_v[1]**2
x1_r = np.pi / abs(par_r[1])
err_x1_r = np.pi * errors_r[1] / par_r[1]**2

print(f"primo minimo verde = ±{x1_v} ± {err_x1_v} mm")
print(f"primo minimo rosso = ±{x1_r} ± {err_x1_r} mm")

dx = x1_r - x1_v
err_dx = np.sqrt(err_x1_r**2 + err_x1_v**2)
print(f"shift del primo minimo (rosso - verde) = {dx} ± {err_dx} mm")

# rapporto delle lunghezze d'onda misurato dai fit 
rapporto = x1_r / x1_v
err_rapporto = rapporto * np.sqrt((err_x1_r / x1_r)**2 + (err_x1_v / x1_v)**2)
rapporto_att = lam_r / lam_v
print(f"lambda_rosso / lambda_verde misurato = {rapporto} ± {err_rapporto}")
print(f"lambda_rosso / lambda_verde atteso   = {rapporto_att}")

if np.abs(rapporto - rapporto_att) < 3*err_rapporto:
    print("il rapporto misurato è compatibile con quello atteso")
else:
    print("il rapporto misurato NON è compatibile con quello atteso")

print("---")

# differenza di lunghezza d'onda corrispondente allo shift [nm]
dlam = lam_v * (rapporto - 1) * 1e6
err_dlam = lam_v * err_rapporto * 1e6
dlam_att = (lam_r - lam_v) * 1e6
print(f"Delta lambda dallo shift = {dlam} ± {err_dlam} nm")
print(f"Delta lambda atteso      = {dlam_att} nm")

if np.abs(dlam - dlam_att) < 3*err_dlam:
    print("Delta lambda è compatibile con quello atteso")
else:
    print("Delta lambda NON è compatibile con quello atteso")

primo minimo verde = ±4.8839089206658 ± 0.002292936725394016 mm
primo minimo rosso = ±6.209518912724084 ± 0.001347533734277342 mm
shift del primo minimo (rosso - verde) = 1.3256099920582836 ± 0.0026595875604454294 mm
lambda_rosso / lambda_verde misurato = 1.2714239789462678 ± 0.0006576012812995217
lambda_rosso / lambda_verde atteso   = 1.2218045112781954
il rapporto misurato NON è compatibile con quello atteso
---
Delta lambda dallo shift = 144.39755679941447 ± 0.34984388165134556 nm
Delta lambda atteso      = 117.99999999999994 nm
Delta lambda NON è compatibile con quello atteso


## Grafico con i due laser

Curve centrate ($x - C$) e scalate ($(V-D)$) cosi resta visibile lo spostamento dei minimi e differenza di voltaggio del max

In [20]:
#  x - C , (V - D)
gl_v = ROOT.TGraphErrors(len(x_v), x_v - par_v[2], (y_v - par_v[3]), err_x_v, err_y_v / par_v[0])
gl_v.SetMarkerStyle(20)
gl_v.SetMarkerSize(0.7)
gl_v.SetMarkerColor(ROOT.kGreen + 2)
gl_v.SetLineColor(ROOT.kGreen + 2)

gl_r = ROOT.TGraphErrors(len(x_r), x_r - par_r[2], (y_r - par_r[3]), err_x_r, err_y_r / par_r[0])
gl_r.SetMarkerStyle(20)
gl_r.SetMarkerSize(0.7)
gl_r.SetMarkerColor(ROOT.kRed + 1)
gl_r.SetLineColor(ROOT.kRed + 1)

mg = ROOT.TMultiGraph()
mg.Add(gl_v, "P")
mg.Add(gl_r, "P")
mg.SetTitle("Confronto tra i due laser; x - C [mm]; (V - D) / A")

fl_v = ROOT.TF1("confronto_v", sinc2, min(x_v) - par_v[2], max(x_v) - par_v[2], 4)
fl_v.SetParameters(par_v[0], par_v[1], 0, 0)
fl_v.SetLineColor(ROOT.kGreen + 2)
fl_v.SetLineWidth(2)

fl_r = ROOT.TF1("confronto_r", sinc2, min(x_r) - par_r[2], max(x_r) - par_r[2], 4)
fl_r.SetParameters(par_r[0], par_r[1], 0, 0)
fl_r.SetLineColor(ROOT.kRed + 1)
fl_r.SetLineWidth(2)

# righe tratteggiate sui primi minimi (±x1)
linee = []
for pos, col in [(x1_v, ROOT.kGreen + 2), (x1_r, ROOT.kRed + 1)]:
    for segno in (+1, -1):
        ln = ROOT.TLine(segno * pos, -0.05, segno * pos, 1)
        ln.SetLineColor(col)
        ln.SetLineStyle(2)
        ln.SetLineWidth(2)
        linee.append(ln)

leg4 = ROOT.TLegend(0.60, 0.70, 0.89, 0.89)
leg4.AddEntry(gl_v, "Dati verde", "pe")
leg4.AddEntry(fl_v, "Fit verde", "l")
leg4.AddEntry(gl_r, "Dati rosso", "pe")
leg4.AddEntry(fl_r, "Fit rosso", "l")

c4 = ROOT.TCanvas("c4", "Confronto laser", 900, 650)
c4.SetGrid()
mg.Draw("A")
fl_v.Draw("same")
fl_r.Draw("same")
for ln in linee:
    ln.Draw()
leg4.Draw()


c4.Draw()
c4.SaveAs("Immagini/confronto_fit.png")

Info in <TCanvas::Print>: png file Immagini/confronto_fit.png has been created


# 2 PARTE

## 2 Fit separati per il rosso

In [21]:
dataset0 = [r[:24] for r in rosso]
dataset1 = [r[24:59] for r in rosso]
dataset2 = [v[:24] for v in verde]
dataset3 = [v[24:59] for v in verde]
dataset = [dataset0, dataset1, dataset2, dataset3]

initialparameters = []

canvases = []
objs = []
fits_r = []
errs_yeff_r = []

for i, (x_r, y_r, err_x_r, err_y_r) in enumerate(dataset):
    # fit preliminare con solo err_y
    pre_r = ROOT.TF1(f"pre_r_{i}", sinc2, min(x_r), max(x_r), 4)
    if i < 2:
        pre_r.SetParameters(y_r.max(), 0.52, -1.0, 0)
    else:
        pre_r.SetParameters(y_v.max(), 0.650, 0.3, 0)
    gr_pre_r = ROOT.TGraphErrors(len(x_r), x_r, y_r, np.zeros_like(x_r), err_y_r)
    gr_pre_r.Fit(pre_r, "SQ0")

    # fit completo con ROOT
    SincFit_r = ROOT.TF1(f"model_r_{i}", sinc2, min(x_r), max(x_r), 4)
    SincFit_r.SetParameters(
        pre_r.GetParameter(0),
        pre_r.GetParameter(1),
        pre_r.GetParameter(2),
        pre_r.GetParameter(3),
    )
    SincFit_r.SetParNames("A", "B", "C", "D")
    if i < 2:
        SincFit_r.SetParLimits(0, 0.0, 1.0)
    else:
        SincFit_r.SetParLimits(0, 0.0, 2.8)
   
    SincFit_r.SetNpx(5000)

    gr1_r = ROOT.TGraphErrors(len(x_r), x_r, y_r, err_x_r, err_y_r)
    gr1_r.SetTitle(f"Dati e fit (rosso) {i+1} parte; x [mm]; V [V]")
    gr1_r.SetMarkerStyle(20)
    gr1_r.SetMarkerSize(0.7)
    gr1_r.SetMarkerColor(ROOT.kAzure + 2)
    gr1_r.SetLineColor(ROOT.kAzure + 3)

    fit_res_r = gr1_r.Fit(SincFit_r, "S")
    print(f"Fit status: {fit_res_r.Status()}")

    par_r = [SincFit_r.GetParameter(k) for k in range(4)]
    errors_r = [SincFit_r.GetParError(k) for k in range(4)]
    chi2_r = fit_res_r.Chi2()
    ndf_r = fit_res_r.Ndf()

    err_yeff_r = np.sqrt(err_y_r**2 + (derivata(x_r, par_r) * err_x_r) ** 2)

    fits_r.append(SincFit_r)
    errs_yeff_r.append(err_yeff_r)

    print("Parametri stimati:")
    print(f"A = {par_r[0]} ± {errors_r[0]}")
    print(f"B = {par_r[1]} ± {errors_r[1]}")
    print(f"C = {par_r[2]} ± {errors_r[2]}")
    print(f"D = {par_r[3]} ± {errors_r[3]}")
    print(f"Chi2 = {chi2_r}, ndf = {ndf_r}")

    c = ROOT.TCanvas(f"c1_r_{i}", f"Fit (rosso) {i}", 800, 600)
    c.SetGrid()
    gr1_r.Draw("AP")

    c.Draw()
    canvases.append(c)
    c.SaveAs(f"fit_parziale_verde_{i}.png")
    objs.extend([gr1_r, SincFit_r, pre_r, gr_pre_r])

Fit status: 0
Parametri stimati:
A = 0.8781254208904022 ± 0.002972795684655527
B = 0.518837309859865 ± 0.0009393996859084833
C = -1.3029268971240047 ± 0.028097724721196725
D = -0.0011157264928056522 ± 6.284864878742116e-05
Chi2 = 492.370646529757, ndf = 20
Fit status: 0
Parametri stimati:
A = 0.8827222762540654 ± 0.003296306063599097
B = 0.5429264407451612 ± 0.000392956604890282
C = -0.6478973705086579 ± 0.006893145704017176
D = -0.000841802896854043 ± 5.0925675681087684e-05
Chi2 = 4467.85452895561, ndf = 22
Fit status: 0
Parametri stimati:
A = 2.7999999999447085 ± 0.004176951330336909
B = 0.6524594349912197 ± 0.0011010574081676842
C = 0.173942194938199 ± 0.025735347383648247
D = 9.826039644020949e-05 ± 7.247574706327558e-05
Chi2 = 142.71993587092206, ndf = 20
Fit status: 0
Parametri stimati:
A = 2.790572975559211 ± 0.012651771144534196
B = 0.6414789396113787 ± 0.0007306747859100374
C = 0.336331055441018 ± 0.018454739807081632
D = -1.739955018959735e-06 ± 7.318085330587339e-05
Chi2 = 4

Info in <TCanvas::Print>: png file fit_parziale_verde_0.png has been created
Info in <TCanvas::Print>: png file fit_parziale_verde_1.png has been created
Info in <TCanvas::Print>: png file fit_parziale_verde_2.png has been created
Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 
Info in <TCanvas::Print>: png file fit_parziale_verde_3.png has been created


In [22]:
canvases_res = []
objs_res = []

for i, (x_r, y_r, err_x_r, err_y_r) in enumerate(dataset):
    mod = fits_r[i]
    err_y_eff = errs_yeff_r[i]

    y_attesi_r = np.array([mod.Eval(x_val) for x_val in x_r])
    residuals_r = y_r - y_attesi_r

    line_r = ROOT.TF1(f"line_r_{i}", retta, min(x_r), max(x_r), 2)
    line_r.SetParameters(0, 0)
    line_r.SetParNames("m", "q")
    line_r.SetLineColor(ROOT.kRed + 1)
    line_r.SetLineWidth(2)

    gr2_r = ROOT.TGraphErrors(len(x_r), x_r, residuals_r, np.zeros_like(x_r), err_y_eff)
    gr2_r.SetTitle(f"Residui in funzione di x (rosso) {i}; x [mm]; V_misurato - V_fit [V]")
    gr2_r.SetMarkerStyle(20)
    gr2_r.SetMarkerSize(0.7)
    gr2_r.SetMarkerColor(ROOT.kAzure + 2)
    gr2_r.SetLineColor(ROOT.kAzure + 3)

    fit_res2_r = gr2_r.Fit(line_r, "S")
    print(f"Fit lineare status ({i}): {fit_res2_r.Status()}")

    par_l_r = [line_r.GetParameter(k) for k in range(2)]
    err_l_r = [line_r.GetParError(k) for k in range(2)]

    c3_r = ROOT.TCanvas(f"c3_r_{i}", f"Residui (rosso) {i}", 800, 600)
    c3_r.SetGrid()

    gr2_r.Draw("AP")

    leg3_r = ROOT.TLegend(0.14, 0.77, 0.42, 0.86)
    leg3_r.AddEntry(gr2_r, "Residui", "pe")
    leg3_r.AddEntry(line_r, "Fit lineare", "l")
    leg3_r.Draw()

    c3_r.Draw()
    canvases_res.append(c3_r)
    objs_res.extend([gr2_r, leg3_r, line_r])

Fit lineare status (0): 0
Fit lineare status (1): 0
Fit lineare status (2): 0
Fit lineare status (3): 0
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      485.275
NDf                       =           22
Edm                       =  1.50713e-09
NCalls                    =           43
m                         =  2.46078e-05   +/-   9.23774e-06 
q                         =   0.00045221   +/-   0.000176375 
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      4263.79
NDf                       =           24
Edm                       =  1.15395e-14
NCalls                    =           43
m                         = -0.000142257   +/-   9.95839e-06 
q                         =   0.00253946   +/-   0.000183811 
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      136.772
NDf                       =           22
Edm                      